In [1]:
# 경고 메시지 무시
import warnings
warnings.filterwarnings(action='ignore') 

import glob
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import csv
import folium
import datetime
import seaborn as sns
import scipy as sp
import statsmodels.formula.api as smf
import networkx as nx
import missingno as msno
import os
import sys
import urllib.request
import time
import json
import plotly.express as px
import re
import sklearn.metrics as metrics
import yfinance as yf
import tensorflow as tf

from keras.utils import to_categorical
from keras.callbacks import ModelCheckpoint,EarlyStopping
from tensorflow.python.keras.utils import np_utils
from keras.datasets import mnist
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from prophet.plot import add_changepoints_to_plot, plot_plotly, plot_components_plotly
from prophet  import Prophet
from openpyxl import load_workbook
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_diabetes
from folium.plugins import HeatMap 
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from dateutil.relativedelta import relativedelta
from sklearn.cluster import KMeans    
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import dendrogram, linkage
from mpl_toolkits.mplot3d import Axes3D
from operator import itemgetter
from PIL import Image
from collections import Counter
from wordcloud import WordCloud
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, ConfusionMatrixDisplay, confusion_matrix, accuracy_score, silhouette_score, classification_report, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier


# plt.rc('font', family='malgun gothic')
# plt.rcParams['axes.unicode_minus']=False  # '- 표시
plt.rc('font',family='D2CodingLigature Nerd Font')

2026-02-24 18:37:58.919812: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# 텍스트마이닝 감성분석

1. 데이터 불러오기
2. 전처리 (정제 및 토큰화 (단어))
3. TF-IDF로 벡터화
4. Logistic Regression 모델 학습
5. 사용자가 입력한 문장의 감성이 긍정인지 부정인지 예측

## 데이터 불러오기 및 전처리

In [2]:
nsmc_train_df = pd.read_csv('../../data/ratings_train.txt', encoding = 'utf8', sep = '\t', engine = 'python') # 학습용
nsmc_train_df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
nsmc_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


### 결측지 제거와 레이블분포 확인(긍정(1)/부정(0))

In [4]:
nsmc_train_df = nsmc_train_df[nsmc_train_df['document'].notnull()]  ## 리뷰가 없는 행 삭제

In [5]:
nsmc_train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB


In [6]:
nsmc_train_df['label'].value_counts()

label
0    75170
1    74825
Name: count, dtype: int64

In [7]:
# 특수 문자, 영어, 숫자 제거 → 한글만 남김
nsmc_train_df['document'] = nsmc_train_df['document'].apply(lambda x : re.sub(r'[^ ㄱ-ㅣ가-힣]+', " ", x))
nsmc_train_df.head()

,id,document,label
0,9976970,아 더빙 진짜 짜증나네요 목소리,0
1,3819312,흠 포스터보고 초딩영화줄 오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 솔직히 재미는 없다 평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


### 평가용 데이터 준비하기

In [8]:
nsmc_test_df = pd.read_csv('../../data/ratings_test.txt', encoding= 'utf8', sep = '\t', engine = 'python') # 평가용
nsmc_test_df.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


### 평가용 데이터에서 결측치 제거 후, 긍정과 부정 레이블 확인하기

In [9]:
nsmc_test_df = nsmc_test_df[nsmc_test_df['document'].notnull()]
nsmc_test_df['label'].value_counts()

label
1    25171
0    24826
Name: count, dtype: int64

### 테스트용 파일에서도 한글 이외 문자 제거

In [10]:
nsmc_test_df['document'] = nsmc_test_df['document'].apply(lambda x : re.sub(r'[^ ㄱ-ㅣ가-힣]+', " ", x))
nsmc_test_df.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,,0
2,8544678,뭐야 이 평점들은 나쁘진 않지만 점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임 돈주고 보기에는,0
4,6723715,만 아니었어도 별 다섯 개 줬을텐데 왜 로 나와서 제 심기를 불편하게 하죠,0


### 한글 형태소(단어)작업을 사용자 정의함수로 만들어 둠

In [12]:
from konlpy.tag import Okt

okt = Okt()

def okt_tokenizer(text):   ## 명사로 분리하는 작업을 사용자정의 함수로 만들어 둠
    tokens = okt.morphs(text)
    return tokens

## TF-IDF 실행  : 실행시간이 5~10분 정도 걸림

In [13]:
# 단어의 중요도를 수치화 (TF-IDF)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(tokenizer=okt_tokenizer, ngram_range=(1, 2), min_df=3, max_df=0.9, token_pattern=None)
tfidf.fit(nsmc_train_df['document'])
nsmc_train_tfidf = tfidf.transform(nsmc_train_df['document'])

* TF-IDF : 자주 나오지만 의미 없는 단어는 낮게, 중요한 단어는 높게 점수 부여 
* ngram_range=(1, 2) : 단어 하나 + 두 단어 묶음
* min_df=3 : 3개 문장 미만에서 등장하는 단어는 제거 (너무 희귀한 단어 삭제)
* max_df=0.9 : 전체 문서의 90% 이상 등장하는 단어 제거 (너무 흔한 단어 삭제)
* token_pattern=None : 정규식으로 토큰 지정 방식을 따른다 (없으면 에러 발생)

* tfidf.fit(nsmc_train_df['document']) : idf값 계산
* tfidf.transform(nsmc_train_df['document']) : 실제로 문서를 TF-IDF 벡터로 변환

## 로지스틱 회귀 모델 생성 및 학습

In [14]:
from sklearn.linear_model import LogisticRegression

SA_lr = LogisticRegression(random_state = 0,  max_iter=500)

In [15]:
SA_lr.fit(nsmc_train_tfidf, nsmc_train_df['label'])

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,0
,solver,'lbfgs'
,max_iter,500
,multi_class,'deprecated'


## 평가용 데이터를 이용하여 모델 정확도 확인하기

In [16]:
#평가용 데이터의 피처 벡터화와 테스트데이터로 예측
nsmc_test_tfidf = tfidf.transform(nsmc_test_df['document'])
test_predict = SA_lr.predict(nsmc_test_tfidf)

In [17]:
from sklearn.metrics import accuracy_score
print('감성 분석 정확도 : ', round(accuracy_score(nsmc_test_df['label'],test_predict), 3))

감성 분석 정확도 :  0.852


In [ ]:
st = input('감성 분석할 문장 입력 >> ') # 오늘은 날씨가 흐려서 기분이 좋지 않았다

In [19]:
#0) 입력 텍스트에 대한 전처리 수행
st = re.compile(r'[ㄱ-ㅣ가-힣]+').findall(st)
print(st)
st = [" ".join(st)]
print(st)

['오늘은', '날씨가', '흐려서', '기분이', '좋지', '않았다']
['오늘은 날씨가 흐려서 기분이 좋지 않았다']


In [21]:
#1) 입력 텍스트의 피처 벡터화
st_tfidf = tfidf.transform(st)

#2) 최적 감성 분석 모델에 적용하여 감성 분석 평가
st_predict = SA_lr.predict(st_tfidf)

In [22]:
#3) 예측값 출력하기
if(st_predict == 0):
    print(st , "->> 부정 감성")
else :
    print(st , "->> 긍정 감성")

['오늘은 날씨가 흐려서 기분이 좋지 않았다'] ->> 부정 감성
